# NDJF Cluster Composites and Representative Events

This notebook turns the validated `k = 3` working subtype framework into interpretation products.

Primary goals:
- load the validated `k = 3` cluster assignments from `Notebook 08` and `Notebook 09`
- pick representative events from each cluster using distance to the cluster centroid in standardized feature space
- link those representative events back to the saved convergence, OLR, and satellite review panels when available
- build peak-time physical composites for the main cluster-interpretation fields
- summarize the cluster story in a form suitable for results writing

Important note:
- this notebook treats `k = 3` as the working subtype framework
- `k = 2` remains the robustness check and `k = 4` remains exploratory finer structure
- the composite fields here are peak-time composites, not full event-window extrema composites


In [ ]:
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git"
BRANCH = "main"
REPO_DIR = "/content/JPCZcatalog"
FORCE_REFRESH_REPO = True
PERSIST_OUTPUTS_TO_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/JPCZcatalog_outputs"

if PERSIST_OUTPUTS_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print("Persistent output dir:", DRIVE_OUTPUT_DIR)

os.chdir("/content")

if FORCE_REFRESH_REPO and os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("Removed existing repo clone:", REPO_DIR)

if not os.path.exists(REPO_DIR):
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
        text=True,
        capture_output=True,
    )
    print(proc.stdout)
    print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{proc.stderr}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-colab.txt"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
        check=True,
    )
else:
    print("Using existing repo clone:", REPO_DIR)

os.chdir(REPO_DIR)
src_dir = os.path.join(REPO_DIR, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print("Working directory:", os.getcwd())


In [ ]:
from pathlib import Path
import shutil

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import pandas as pd
import xarray as xr

from jpcz_catalog.analysis import add_japan_local_time_columns
from jpcz_catalog.config import (
    COASTAL_JAPAN_BOX,
    HOKKAIDO_BOX,
    HOKKAIDO_FRONT_BOX,
    JPCZ_POLYGON_VERTICES,
    OBJECTIVE_SUBTYPE_DOMAIN,
    PACIFIC_EAST_OF_JAPAN_BOX,
    PACIFIC_FRONT_BOX,
    SEA_OF_JAPAN_BOX,
)
from jpcz_catalog.detect import compute_divergence_field, prepare_detection_geometry
from jpcz_catalog.diagnostics import (
    compute_geopotential_height_field,
    compute_relative_vorticity_field,
    compute_temperature_gradient_magnitude,
    load_snapshot,
)
from jpcz_catalog.era5 import open_arco_era5
from jpcz_catalog.satellite import default_modis_layers_for_date
from jpcz_catalog.subtypes import feature_definitions_dataframe, standardize_feature_table

RUN_EXPORT_DIR_08 = Path("outputs/verification/objective_subtype_runs")
RUN_EXPORT_DIR_09 = Path("outputs/verification/objective_subtype_validation")
COMPOSITE_EXPORT_DIR = Path("outputs/verification/objective_subtype_cluster_examples")
COMPOSITE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR = Path("outputs/verification/objective_subtype_cluster_example_plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
CLIMATOLOGY_PATH = Path("outputs/verification/z850_ndjf_monthly_climatology.nc")
FEATURE_TABLE_PATH = Path("outputs/verification/jpcz_catalog_ndjf_objective_subtype_features.csv")

QUICKLOOK_DIR = Path("outputs/verification/ndjf_event_quicklooks_merged_12h")
SATELLITE_DIR = Path("outputs/verification/ndjf_event_satellite_panels_merged_12h")
OLR_DIR = Path("outputs/verification/ndjf_event_olr_panels_merged_12h")

PRIMARY_CLUSTER_K = 3
PRIMARY_CLUSTER_COLUMN = f"cluster_ward_{PRIMARY_CLUSTER_K}"
REPRESENTATIVE_EVENTS_PER_CLUSTER = 5
DISPLAY_REPRESENTATIVE_EVENTS_PER_CLUSTER = 1
SAVE_PLOTS = True
FORCE_REBUILD_CLUSTER_COMPOSITES = False
ERA5_TIME_CHUNK = 48
PROGRESS_EVERY = 10

CLUSTER_FEATURE_COLUMNS = [
    "coastal_to_jpcz_mean_convergence_ratio",
    "hokkaido_min_z850_anomaly_tminus12_to_tplus12",
    "front_box_max_temp_gradient_850_tminus12_to_tplus12",
    "sea_of_japan_mean_vorticity_peak_925",
]

CLUSTER_LABELS = {
    1: "Cluster 1: least synoptic / weaker-background",
    2: "Cluster 2: moderately synoptic / circulation-modified",
    3: "Cluster 3: strongly synoptic / frontal / coastal-enhanced",
}

FEATURE_DICTIONARY = feature_definitions_dataframe()
FEATURE_UNITS = FEATURE_DICTIONARY.set_index("column_name")["units"].to_dict()


def maybe_copy_to_drive(path: Path):
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if path.is_file():
        shutil.copy2(path, drive_path)
        print("Copied to Drive:", drive_path)


def restore_from_drive_cache(path: Path) -> bool:
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return False
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if not drive_path.exists():
        return False
    path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(drive_path, path)
    print("Restored from Drive:", drive_path, "->", path)
    return True


def axis_label(column_name: str) -> str:
    units = FEATURE_UNITS.get(column_name)
    if units is None or units == "unitless":
        return column_name
    return f"{column_name}\n[{units}]"


def quicklook_name_for_row(row_index: int, row: pd.Series) -> str:
    return f"{pd.Timestamp(row['event_peak']).strftime('%Y%m%dT%H%M')}_idx{row_index:03d}.png"


def satellite_name_for_row(row_index: int, row: pd.Series) -> str | None:
    satellite_layers = default_modis_layers_for_date(pd.Timestamp(row['event_peak']).normalize())
    if not satellite_layers:
        return None
    satellite_layer = satellite_layers[0]
    layer_slug = (
        satellite_layer.replace("MODIS_", "")
        .replace("_CorrectedReflectance_TrueColor", "")
        .lower()
    )
    return f"{pd.Timestamp(row['event_peak']).strftime('%Y%m%dT%H%M')}_idx{row_index:03d}_{layer_slug}.jpg"


def ensure_local_copy(local_path: Path, drive_subdir_name: str) -> bool:
    if local_path.exists():
        return True
    drive_path = Path(DRIVE_OUTPUT_DIR) / drive_subdir_name / local_path.name
    if not drive_path.exists():
        return False
    local_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(drive_path, local_path)
    return True


In [ ]:
paths_to_restore = [
    FEATURE_TABLE_PATH,
    CLIMATOLOGY_PATH,
    RUN_EXPORT_DIR_08 / "clustered_events_k3.csv",
    RUN_EXPORT_DIR_08 / "cluster_medians_k3.csv",
    RUN_EXPORT_DIR_08 / "cluster_counts_k3.csv",
    RUN_EXPORT_DIR_08 / "cluster_comparison_summary.csv",
    RUN_EXPORT_DIR_09 / "cluster_validation_summary.csv",
    RUN_EXPORT_DIR_09 / "cluster_nesting_k2_vs_k3_counts.csv",
    RUN_EXPORT_DIR_09 / "cluster_nesting_k2_vs_k3_row_fraction.csv",
    RUN_EXPORT_DIR_09 / "pairwise_cluster_checks_k3.csv",
]
for path in paths_to_restore:
    if not path.exists():
        restore_from_drive_cache(path)

clustered_k3_df = pd.read_csv(RUN_EXPORT_DIR_08 / "clustered_events_k3.csv")
clustered_k3_df = add_japan_local_time_columns(clustered_k3_df)
if PRIMARY_CLUSTER_COLUMN not in clustered_k3_df.columns:
    cluster_cols = [c for c in clustered_k3_df.columns if c.startswith("cluster_")]
    raise RuntimeError(f"Expected {PRIMARY_CLUSTER_COLUMN} in clustered_events_k3.csv, found {cluster_cols}")

cluster_counts_path = RUN_EXPORT_DIR_08 / "cluster_counts_k3.csv"
cluster_medians_path = RUN_EXPORT_DIR_08 / "cluster_medians_k3.csv"
if cluster_counts_path.exists():
    cluster_counts_df = pd.read_csv(cluster_counts_path, index_col=0)
else:
    cluster_counts_df = clustered_k3_df[PRIMARY_CLUSTER_COLUMN].value_counts().sort_index().rename("n_events").to_frame()
if cluster_medians_path.exists():
    cluster_medians_df = pd.read_csv(cluster_medians_path, index_col=0)
else:
    cluster_medians_df = clustered_k3_df.groupby(PRIMARY_CLUSTER_COLUMN).median(numeric_only=True).round(2)
validation_summary_df = pd.read_csv(RUN_EXPORT_DIR_09 / "cluster_validation_summary.csv")
k2_vs_k3_counts_df = pd.read_csv(RUN_EXPORT_DIR_09 / "cluster_nesting_k2_vs_k3_counts.csv", index_col=0)
k2_vs_k3_fraction_df = pd.read_csv(RUN_EXPORT_DIR_09 / "cluster_nesting_k2_vs_k3_row_fraction.csv", index_col=0)
pairwise_k3_df = pd.read_csv(RUN_EXPORT_DIR_09 / "pairwise_cluster_checks_k3.csv")

print("Loaded k=3 working subtype outputs")
display(cluster_counts_df)
print("\nValidation summary row for k=3")
display(validation_summary_df.loc[validation_summary_df["k"] == 3])
print("\nk=2 vs k=3 nesting counts")
display(k2_vs_k3_counts_df)
print("\nk=2 vs k=3 nesting row fractions")
display(k2_vs_k3_fraction_df)


In [ ]:
standardized_df, feature_means, feature_stds = standardize_feature_table(
    clustered_k3_df.copy(),
    columns=CLUSTER_FEATURE_COLUMNS,
)
cluster_labels = clustered_k3_df[PRIMARY_CLUSTER_COLUMN]
valid_mask = standardized_df.notna().all(axis=1) & cluster_labels.notna()
standardized_valid = standardized_df.loc[valid_mask].copy()
labels_valid = cluster_labels.loc[valid_mask].astype(int)
centroids = standardized_valid.groupby(labels_valid).mean()

representative_records = []
for cluster_id in sorted(centroids.index):
    member_index = labels_valid.index[labels_valid == cluster_id]
    member_points = standardized_valid.loc[member_index]
    centroid = centroids.loc[cluster_id]
    distances = np.sqrt(((member_points - centroid) ** 2).sum(axis=1))
    top_members = distances.sort_values().head(REPRESENTATIVE_EVENTS_PER_CLUSTER)

    for rank_within_cluster, (row_index, centroid_distance) in enumerate(top_members.items(), start=1):
        row = clustered_k3_df.loc[row_index].copy()
        quicklook_name = quicklook_name_for_row(row_index, row)
        satellite_name = satellite_name_for_row(row_index, row)
        olr_name = quicklook_name
        quicklook_exists = ((QUICKLOOK_DIR / quicklook_name).exists() or (Path(DRIVE_OUTPUT_DIR) / QUICKLOOK_DIR.name / quicklook_name).exists())
        olr_exists = ((OLR_DIR / olr_name).exists() or (Path(DRIVE_OUTPUT_DIR) / OLR_DIR.name / olr_name).exists())
        satellite_exists = False if satellite_name is None else ((SATELLITE_DIR / satellite_name).exists() or (Path(DRIVE_OUTPUT_DIR) / SATELLITE_DIR.name / satellite_name).exists())
        representative_records.append(
            {
                "cluster_id": int(cluster_id),
                "cluster_label": CLUSTER_LABELS.get(int(cluster_id), f"Cluster {int(cluster_id)}"),
                "representative_rank": int(rank_within_cluster),
                "catalog_index": int(row_index),
                "centroid_distance": float(centroid_distance),
                "event_peak_utc": pd.Timestamp(row["event_peak"]),
                "event_peak_jst": pd.Timestamp(row["event_peak_jst"]) if pd.notna(row.get("event_peak_jst")) else pd.NaT,
                "duration_hours": float(row["duration_hours"]),
                "event_peak_D_1e5_s-1": float(row["event_peak_D_1e5_s-1"]),
                "coastal_to_jpcz_mean_convergence_ratio": float(row["coastal_to_jpcz_mean_convergence_ratio"]),
                "sea_of_japan_mean_vorticity_peak_925": float(row["sea_of_japan_mean_vorticity_peak_925"]),
                "hokkaido_min_z850_anomaly_tminus12_to_tplus12": float(row["hokkaido_min_z850_anomaly_tminus12_to_tplus12"]),
                "front_box_max_temp_gradient_850_tminus12_to_tplus12": float(row["front_box_max_temp_gradient_850_tminus12_to_tplus12"]),
                "pattern_type_manual": row.get("pattern_type_manual", ""),
                "verification_notes": row.get("verification_notes", ""),
                "quicklook_name": quicklook_name,
                "olr_name": olr_name,
                "satellite_name": satellite_name,
                "quicklook_exists": bool(quicklook_exists),
                "olr_exists": bool(olr_exists),
                "satellite_exists": bool(satellite_exists),
            }
        )

representative_events_df = pd.DataFrame(representative_records).sort_values(["cluster_id", "representative_rank"]).reset_index(drop=True)
representatives_path = COMPOSITE_EXPORT_DIR / "k3_representative_events.csv"
representative_events_df.to_csv(representatives_path, index=False)
maybe_copy_to_drive(representatives_path)

cluster_profile_df = cluster_medians_df.copy().reset_index().rename(columns={cluster_medians_df.index.name or PRIMARY_CLUSTER_COLUMN: "cluster_id"})
cluster_profile_df["cluster_id"] = cluster_profile_df["cluster_id"].astype(int)
cluster_profile_df.insert(1, "cluster_label", cluster_profile_df["cluster_id"].map(CLUSTER_LABELS))
cluster_profile_path = COMPOSITE_EXPORT_DIR / "k3_cluster_profile_medians.csv"
cluster_profile_df.to_csv(cluster_profile_path, index=False)
maybe_copy_to_drive(cluster_profile_path)

print("Representative events nearest the k=3 cluster centroids")
display(representative_events_df)
print("\nCluster median profile table for k=3")
display(cluster_profile_df)


In [ ]:
def display_representative_panels(representative_df: pd.DataFrame, *, per_cluster: int = 1):
    rows_to_show = representative_df.groupby("cluster_id", group_keys=False).head(per_cluster)
    for _, rep in rows_to_show.iterrows():
        print(rep["cluster_label"])
        print(
            f"catalog idx {int(rep['catalog_index'])} | UTC {pd.Timestamp(rep['event_peak_utc']):%Y-%m-%d %H:%M} | "
            f"JST {pd.Timestamp(rep['event_peak_jst']):%Y-%m-%d %H:%M} | centroid distance {rep['centroid_distance']:.3f}"
        )

        panel_specs = [
            ("Convergence quicklook", QUICKLOOK_DIR / rep["quicklook_name"], QUICKLOOK_DIR.name),
            ("OLR panel", OLR_DIR / rep["olr_name"], OLR_DIR.name),
        ]
        if pd.notna(rep["satellite_name"]) and rep["satellite_name"]:
            panel_specs.append(("Satellite panel", SATELLITE_DIR / rep["satellite_name"], SATELLITE_DIR.name))

        fig, axes = plt.subplots(1, len(panel_specs), figsize=(6 * len(panel_specs), 6))
        if len(panel_specs) == 1:
            axes = [axes]

        for ax, (panel_label, local_path, drive_subdir_name) in zip(axes, panel_specs):
            has_panel = ensure_local_copy(local_path, drive_subdir_name)
            if not has_panel:
                ax.axis("off")
                ax.set_title(f"{panel_label}\nmissing")
                continue
            image = mpimg.imread(local_path)
            ax.imshow(image)
            ax.axis("off")
            ax.set_title(panel_label)

        fig.suptitle(rep["cluster_label"], y=0.98)
        fig.tight_layout()
        plt.show()


display_representative_panels(
    representative_events_df,
    per_cluster=DISPLAY_REPRESENTATIVE_EVENTS_PER_CLUSTER,
)


In [ ]:
if not CLIMATOLOGY_PATH.exists():
    restore_from_drive_cache(CLIMATOLOGY_PATH)
z850_climatology = xr.open_dataarray(CLIMATOLOGY_PATH).load()

composite_dataset_path = COMPOSITE_EXPORT_DIR / "k3_peak_time_cluster_composites.nc"
composite_count_path = COMPOSITE_EXPORT_DIR / "k3_peak_time_cluster_composite_counts.csv"

if composite_dataset_path.exists() and not FORCE_REBUILD_CLUSTER_COMPOSITES:
    composite_ds = xr.open_dataset(composite_dataset_path).load()
    if composite_count_path.exists():
        composite_counts_df = pd.read_csv(composite_count_path)
    else:
        composite_counts_df = pd.DataFrame(
            {
                "cluster_id": sorted(clustered_k3_df[PRIMARY_CLUSTER_COLUMN].dropna().astype(int).unique()),
                "cluster_label": [CLUSTER_LABELS[int(cluster_id)] for cluster_id in sorted(clustered_k3_df[PRIMARY_CLUSTER_COLUMN].dropna().astype(int).unique())],
                "n_events": [int((clustered_k3_df[PRIMARY_CLUSTER_COLUMN] == cluster_id).sum()) for cluster_id in sorted(clustered_k3_df[PRIMARY_CLUSTER_COLUMN].dropna().astype(int).unique())],
            }
        )
    print("Loaded cached k=3 peak-time cluster composites")
else:
    ds = open_arco_era5(chunks={"time": ERA5_TIME_CHUNK})
    geometry_925 = None
    composite_sums = {}
    composite_counts = {}

    for cluster_id in sorted(clustered_k3_df[PRIMARY_CLUSTER_COLUMN].dropna().astype(int).unique()):
        cluster_members = clustered_k3_df.loc[clustered_k3_df[PRIMARY_CLUSTER_COLUMN] == cluster_id].copy()
        composite_counts[cluster_id] = len(cluster_members)
        field_sums = None

        for event_number, (_, row) in enumerate(cluster_members.iterrows(), start=1):
            peak_snapshot_925 = load_snapshot(
                ds,
                row["event_peak"],
                variables=("u_component_of_wind", "v_component_of_wind"),
                domain=OBJECTIVE_SUBTYPE_DOMAIN,
                level=925,
            )
            if geometry_925 is None:
                geometry_925 = prepare_detection_geometry(
                    peak_snapshot_925.longitude,
                    peak_snapshot_925.latitude,
                    JPCZ_POLYGON_VERTICES,
                )

            divergence_field = compute_divergence_field(
                peak_snapshot_925,
                dx=geometry_925.dx,
                dy=geometry_925.dy,
            )
            convergence_field = ((-divergence_field).clip(min=0.0) * 1e5).rename("convergence_925_display")
            vorticity_field = (
                compute_relative_vorticity_field(
                    peak_snapshot_925,
                    dx=geometry_925.dx,
                    dy=geometry_925.dy,
                )
                * 1e5
            ).rename("relative_vorticity_925_display")

            synoptic_snapshot_850 = load_snapshot(
                ds,
                row["event_peak"],
                variables=("geopotential", "temperature"),
                domain=OBJECTIVE_SUBTYPE_DOMAIN,
                level=850,
            )
            z850 = compute_geopotential_height_field(synoptic_snapshot_850)
            z850_anomaly = (z850 - z850_climatology.sel(month=pd.Timestamp(row["event_peak"]).month)).rename("z850_anomaly_t0")
            temp_grad = compute_temperature_gradient_magnitude(synoptic_snapshot_850)
            temp_grad_display = (
                temp_grad * float(temp_grad.attrs.get("display_scale_factor", 1.0))
            ).rename("temperature_gradient_850_display_t0")

            event_ds = xr.Dataset(
                {
                    "convergence_925_display": convergence_field,
                    "relative_vorticity_925_display": vorticity_field,
                    "z850_anomaly_t0": z850_anomaly,
                    "temperature_gradient_850_display_t0": temp_grad_display,
                }
            )

            if field_sums is None:
                field_sums = event_ds.load()
            else:
                field_sums = (field_sums + event_ds).load()

            if PROGRESS_EVERY > 0 and (event_number % PROGRESS_EVERY == 0 or event_number == len(cluster_members)):
                print(
                    f"Composite accumulation for cluster {cluster_id}: {event_number}/{len(cluster_members)} events"
                )

        composite_sums[cluster_id] = (field_sums / len(cluster_members)).expand_dims(cluster_id=[cluster_id])

    composite_ds = xr.concat(
        [composite_sums[cluster_id] for cluster_id in sorted(composite_sums)],
        dim="cluster_id",
    )
    composite_counts_df = pd.DataFrame(
        {
            "cluster_id": sorted(composite_counts),
            "cluster_label": [CLUSTER_LABELS[cluster_id] for cluster_id in sorted(composite_counts)],
            "n_events": [composite_counts[cluster_id] for cluster_id in sorted(composite_counts)],
        }
    )
    composite_ds.to_netcdf(composite_dataset_path)
    composite_counts_df.to_csv(composite_count_path, index=False)
    maybe_copy_to_drive(composite_dataset_path)
    maybe_copy_to_drive(composite_count_path)

print("Peak-time composite counts")
display(composite_counts_df)


In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.patches import Polygon
from jpcz_catalog.masks import box_outline

field_specs = [
    {
        "field": "convergence_925_display",
        "title": "925 hPa convergence composite",
        "cmap": "YlOrRd",
        "levels": np.arange(0, max(6.5, float(composite_ds["convergence_925_display"].max()) + 0.5), 0.5),
        "extend": "max",
        "cbar": "Convergence [1e-5 s^-1]",
    },
    {
        "field": "relative_vorticity_925_display",
        "title": "925 hPa relative vorticity composite",
        "cmap": "RdBu_r",
        "levels": np.arange(-2.5, 4.1, 0.5),
        "extend": "both",
        "cbar": "Relative vorticity [1e-5 s^-1]",
    },
    {
        "field": "z850_anomaly_t0",
        "title": "850 hPa geopotential-height anomaly composite",
        "cmap": "RdBu_r",
        "levels": np.arange(-220, 221, 20),
        "extend": "both",
        "cbar": "Z850 anomaly [gpm]",
    },
    {
        "field": "temperature_gradient_850_display_t0",
        "title": "850 hPa temperature-gradient magnitude composite",
        "cmap": "magma",
        "levels": np.arange(0, max(30, float(composite_ds["temperature_gradient_850_display_t0"].max()) + 2), 2),
        "extend": "max",
        "cbar": "|grad T850| [K (100 km)^-1]",
    },
]

boxes = {
    "Coastal Japan": COASTAL_JAPAN_BOX,
    "Pacific east of Japan": PACIFIC_EAST_OF_JAPAN_BOX,
    "Hokkaido": HOKKAIDO_BOX,
    "Sea of Japan": SEA_OF_JAPAN_BOX,
    "Hokkaido front": HOKKAIDO_FRONT_BOX,
    "Pacific front": PACIFIC_FRONT_BOX,
}

fig, axes = plt.subplots(
    nrows=len(composite_ds["cluster_id"]),
    ncols=len(field_specs),
    figsize=(5.5 * len(field_specs), 4.4 * len(composite_ds["cluster_id"])),
    subplot_kw={"projection": ccrs.PlateCarree()},
)
if len(composite_ds["cluster_id"]) == 1:
    axes = np.expand_dims(axes, axis=0)

for row_idx, cluster_id in enumerate(composite_ds["cluster_id"].values.tolist()):
    for col_idx, spec in enumerate(field_specs):
        ax = axes[row_idx, col_idx]
        ax.set_extent(
            [OBJECTIVE_SUBTYPE_DOMAIN.lon_min, OBJECTIVE_SUBTYPE_DOMAIN.lon_max, OBJECTIVE_SUBTYPE_DOMAIN.lat_min, OBJECTIVE_SUBTYPE_DOMAIN.lat_max],
            crs=ccrs.PlateCarree(),
        )
        ax.coastlines(resolution="50m", linewidth=0.9)
        ax.add_feature(cfeature.BORDERS, linewidth=0.5)
        ax.add_feature(cfeature.LAND, facecolor="lightgray", alpha=0.35)

        field = composite_ds[spec["field"]].sel(cluster_id=cluster_id)
        cf = ax.contourf(
            field.longitude,
            field.latitude,
            field,
            levels=spec["levels"],
            cmap=spec["cmap"],
            extend=spec["extend"],
            transform=ccrs.PlateCarree(),
        )

        polygon = Polygon(
            JPCZ_POLYGON_VERTICES,
            closed=True,
            fill=False,
            edgecolor="black",
            linewidth=1.8,
            transform=ccrs.PlateCarree(),
        )
        ax.add_patch(polygon)

        for box_idx, (label, box) in enumerate(boxes.items()):
            rect_lon, rect_lat = box_outline(box)
            ax.plot(
                rect_lon,
                rect_lat,
                linestyle="--",
                linewidth=1.0,
                color=("navy", "darkgreen", "darkorange", "purple", "crimson", "teal")[box_idx % 6],
                transform=ccrs.PlateCarree(),
            )

        gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4)
        gl.top_labels = False
        gl.right_labels = False
        if col_idx > 0:
            gl.left_labels = False
        if row_idx < len(composite_ds["cluster_id"]) - 1:
            gl.bottom_labels = False

        if row_idx == 0:
            ax.set_title(spec["title"])
        if col_idx == 0:
            ax.text(
                -0.12,
                0.5,
                CLUSTER_LABELS[int(cluster_id)],
                transform=ax.transAxes,
                rotation=90,
                va="center",
                ha="center",
                fontsize=10,
                fontweight="bold",
            )

        cbar = fig.colorbar(cf, ax=ax, orientation="horizontal", fraction=0.046, pad=0.05)
        cbar.set_label(spec["cbar"])

fig.suptitle("k=3 peak-time cluster composites for interpretation", y=0.995)
fig.tight_layout(rect=(0, 0, 1, 0.985))
composite_plot_path = PLOT_DIR / "k3_peak_time_cluster_composites.png"
if SAVE_PLOTS:
    fig.savefig(composite_plot_path, dpi=170, bbox_inches="tight")
    maybe_copy_to_drive(composite_plot_path)
plt.show()


In [ ]:
summary_rows = []
for cluster_id in sorted(clustered_k3_df[PRIMARY_CLUSTER_COLUMN].dropna().astype(int).unique()):
    subset = clustered_k3_df.loc[clustered_k3_df[PRIMARY_CLUSTER_COLUMN] == cluster_id].copy()
    summary_rows.append(
        {
            "cluster_id": int(cluster_id),
            "cluster_label": CLUSTER_LABELS[int(cluster_id)],
            "n_events": int(len(subset)),
            "median_coastal_ratio": float(subset["coastal_to_jpcz_mean_convergence_ratio"].median()),
            "median_hokkaido_z850_anomaly": float(subset["hokkaido_min_z850_anomaly_tminus12_to_tplus12"].median()),
            "median_frontality": float(subset["front_box_max_temp_gradient_850_tminus12_to_tplus12"].median()),
            "median_soj_vorticity": float(subset["sea_of_japan_mean_vorticity_peak_925"].median()),
            "median_polygon_peak_convergence": float(subset["jpcz_polygon_max_convergence_peak_925"].median()),
            "representative_catalog_indices": ", ".join(
                str(int(value))
                for value in representative_events_df.loc[
                    representative_events_df["cluster_id"] == cluster_id,
                    "catalog_index",
                ].head(3)
            ),
        }
    )

cluster_story_df = pd.DataFrame(summary_rows)
cluster_story_path = COMPOSITE_EXPORT_DIR / "k3_cluster_story_summary.csv"
cluster_story_df.to_csv(cluster_story_path, index=False)
maybe_copy_to_drive(cluster_story_path)

print("k=3 interpretation summary")
display(cluster_story_df)
print("\nUse this notebook together with Notebook 09 like this:")
print("- Notebook 09 justifies k=3 statistically as the working subtype framework.")
print("- Notebook 10 shows what those three clusters look like physically and which events are most representative of each cluster.")
